# 06 Evaluation

## Objective

Evaluate retrieval quality and answer quality of the RAG pipeline.

## Goals

- Evaluate chunk retrieval
- Evaluate generated answers
- Analyze retrieval distances
- Measure answer length

## Use Case

Fraud Detection Knowledge Assistant

In [8]:
!pip install pypdf pandas sentence-transformers faiss-cpu transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.5 MB/s eta 0:00:00


In [9]:
# Evaluation Questions

evaluation_questions = [
    "How does the paper detect credit card fraud?",
    "What dataset is used in the study?",
    "How many fraudulent transactions are contained in the dataset?",
    "What machine learning models are discussed?",
    "What are the main contributions of the paper?"
]

evaluation_questions

['How does the paper detect credit card fraud?',
 'What dataset is used in the study?',
 'How many fraudulent transactions are contained in the dataset?',
 'What machine learning models are discussed?',
 'What are the main contributions of the paper?']

In [10]:
import faiss
import pandas as pd

from pathlib import Path
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from transformers import pipeline

# PDF Text Extraction Function
def extract_pdf_text(pdf_path):
  reader = PdfReader(pdf_path)
  text = ""
  for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
      text += page_text + "\n"
  return text

# Text Chunking Function
def create_chunks(text, chunk_size=1000):
  chunks = []
  for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i + chunk_size])
  return chunks

# Load Documents
RAW_DATA_DIR = Path("data/raw")
pdf_files = list(RAW_DATA_DIR.glob("*.pdf"))
documents = []
for pdf_file in pdf_files:
  text = extract_pdf_text(pdf_file)
  documents.append({"file_name": pdf_file.name, "text": text})
documents_df = pd.DataFrame(documents)

# Create Chunks
chunk_records = []
for _, row in documents_df.iterrows():
  chunks = create_chunks(row["text"])
  for idx, chunk in enumerate(chunks):
    chunk_records.append({"file_name": row["file_name"], "chunk_id": idx, "chunk_text": chunk})
chunks_df = pd.DataFrame(chunk_records)
print(f"Total chunks: {len(chunks_df)}")

# Load Embedding Model
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")

# Generate Embeddings
embeddings = model.encode(chunks_df["chunk_text"].tolist(), show_progress_bar=True)

# Build FAISS Index
embedding_dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dimension)
index.add(embeddings)
print(f"Vectors stored: {index.ntotal}")

# Retriever Function
def retrieve_context(question, k=3):
  query_embedding = model.encode([question])
  distances, indices = index.search(query_embedding, k)
  context = []
  for idx in indices[0]:
    context.append(chunks_df.iloc[idx]["chunk_text"])
  return "\n\n".join(context)

# LLM laden
generator = pipeline("text-generation", model="distilgpt2")
print("LLM loaded.")

# RAG Function
def ask_question(question):
  if chunks_df.empty:
    raise ValueError("No chunks available.")

  context = retrieve_context(question)
  prompt = f"""
  Context:

  {context}

  Question:

  {question}

  Answer:
  """
  answer = generator(prompt, max_new_tokens=200)
  return answer[0]["generated_text"]

# Evaluate Retieval
for question in evaluation_questions:
    print("=" * 80)
    print(f"Question: {question}")
    query_embedding = model.encode([question])
    distances, indices = index.search(query_embedding, k=3)
    print(f"Average distance "
    f"{distances.mean():.4f}")
    print(f"Retrieved chunks: "
    f"{indices[0]}")

# Evaluate Answers
for question in evaluation_questions:
  print("=" * 80)
  print(f"Question: {question}")
  answer = ask_question(question)
  print()
  print("Answer:")
  print(answer)
  print()

# Answer length Analysis
results =[]
for question in evaluation_questions:
  answer = ask_question(question)
  results.append({"question": question,
                 "answer_length": len(answer)})
evaluation_df = pd.DataFrame(results)
evaluation_df

# Retrieval Distance Analysis
distance_results = []
for question in evaluation_questions:
  query_embedding = model.encode([question])
  distances, indices = index.search(query_embedding, k=3)
  distance_results.append({"question": question,
                           "avg_distance": distances.mean()})
distance_df = pd.DataFrame(distance_results)
distance_df

# Best Retrieval Results
distance_df.sort_values(by="avg_distance")



Total chunks: 88


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


Answer:

  Context:

  thms Gradient Boosting, XGBoost, LightGBM, CatBoost, AdaBoost with Decision Tree base, AdaBoost with Extra Trees base, 
AdaBoost with Logistic Regression base, AdaBoost with Random Forest base, AdaBoost with Ridge Classifier base
Bagging algorithms Bagging Classifier with Decision Trees
Neural networks Neural Network, Artificial Neural Network (ANN), Feedforward Neural Network (FFNN), Multilayer Perceptron (MLP), 
Convolutional Neural Network (CNN)
 Recurrent neural networks Recurrent Neural Network (RNN), Long Short-Term Memory (LSTM), Gated Recurrent Unit (GRU), Bidirectional LSTM 
(BiLSTM), Bidirectional GRU (BiGRU), BiLSTM with Maxpooling, BiGRU with Maxpooling
 Ensemble methods Stacking Classifier (RF , CNN, LSTM, XGBoost, LogisticRegression as meta-learner), Stacking Classifier (RF , CNN, LSTM, 
XGBoost as meta-learner), Stacking Classifier (CNN, LSTM, Transformer Base Learners, XGBoost as meta-learner)
 Proposed models Stacking Classifier (ET, CNN, LSTM, 

[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:

  Context:

  e, unless indicated otherwise in a credit line to the material. If material is not included 
in the article’s Creative Commons licence and your intended use is not permitted by statutory regulation or 
exceeds the permitted use, you will need to obtain permission directly from the copyright holder. To view a copy 
of this licence, visit http://creativecommons.org/licenses/by/4.0/.
© The Author(s) 2026 
Scientific Reports |        (2026) 16:10944 36| https://doi.org/10.1038/s41598-026-42891-4
www.nature.com/scientificreports/


hniques, providing insights into 
its decision-making process and feature importance patterns.
LIME analysis
The LIME analysis reveals the hierarchical decision-making process of Model 1, as illustrated in Figure 13. The 
model achieves perfect classification confidence with prediction probabilities of 1.00 for Not Fraud and 0.00 for 
Fraud for the analyzed instance. The decision tree structure demonstrates clear logical progression throug

[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

,question,avg_distance
0,How does the paper detect credit card fraud?,0.890608
2,How many fraudulent transactions are contained...,0.953671
3,What machine learning models are discussed?,0.971029
1,What dataset is used in the study?,1.429055
4,What are the main contributions of the paper?,1.510020


## Conclusion

The evaluation shows how well the retrieval component and language model perform.

Future improvements include:

- Better embedding models
- Better LLMs
- Larger document collections
- Multi-document retrieval
- RAG evaluation metrics

In [12]:
!git status

Refresh index: 100% (8/8), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/05_rag_pipeline.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/06_evaluation.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
